# 04 — 閉ループと2つの時間刻み

MuJoCoの細かい周期とMPCの遅い周期、receding horizonの関係を分離します。

**前提**: `03_mujoco_go2_plant.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## Multi-rate control

- simulator: \(\Delta t_{sim}=0.002\,s\)（500 Hz）
- MPC呼出し: 100 Hz
- MPC予測刻み: \(\Delta t_{mpc}=0.02\,s\)
- horizon: \(N=12\)、予測時間 \(T=N\Delta t=0.24\,s\)

予測刻みと呼出し周期は同じとは限りません。MPCは0.24秒先まで予測しつつ、
0.01秒後には新しい観測で解き直します。

In [2]:
from quadruped_pympc import config as cfg
sim_dt = cfg.simulation_params["dt"]
mpc_call_hz = cfg.simulation_params["mpc_frequency"]
stride = round(1 / (mpc_call_hz * sim_dt))
prediction = cfg.mpc_params["horizon"] * cfg.mpc_params["dt"]
print("simulation Hz :", 1/sim_dt)
print("MPC every     :", stride, "sim steps")
print("MPC call Hz   :", 1/(stride*sim_dt))
print("horizon time  :", prediction, "s")
assert stride == 5

simulation Hz : 500.0
MPC every     : 5 sim steps
MPC call Hz   : 100.0
horizon time  : 0.24 s


## 1周期の擬似コード

1. MuJoCoから状態を読む
2. gait phaseを `dt_sim` だけ進める
3. contact sequenceとfootholdを作る
4. 5 stepごとにOCPを解き、先頭入力を保存する
5. 毎step、保存GRFと遊脚軌道からトルクを計算する
6. clipし、MuJoCoを1 step進める

この非同期性により、MPCのGRFは最大約10 ms保持されます。
高速振動が見えたとき、予測刻みだけでなく呼出し周期も調査対象です。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。